# WildDex Model Training
**EfficientNetB0 transfer learning — 200 species**

Steps:
1. Install deps
2. Download images from iNaturalist (~200/class)
3. Train EfficientNetB0 (2-phase)
4. Convert to TFLite (float16)
5. Download `wilddex_model.tflite` + `labels.txt`

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q requests tensorflow

In [ ]:
# ── 2. Define 200 species ───────────────────────────────────────────────────
# Keys = folder names (become model labels, sorted alphabetically)
# Values = iNaturalist taxon name (scientific or common)

INAT_NAMES = {
    'alligator':               'Alligator mississippiensis',
    'american_goldfinch':      'Spinus tristis',
    'american_robin':          'Turdus migratorius',
    'anaconda':                'Eunectes murinus',
    'armadillo':               'Dasypus novemcinctus',
    'axolotl':                 'Ambystoma mexicanum',
    'baboon':                  'Papio',
    'bald_eagle':              'Haliaeetus leucocephalus',
    'barn_owl':                'Tyto alba',
    'barn_swallow':            'Hirundo rustica',
    'bat':                     'Chiroptera',
    'bearded_dragon':          'Pogona vitticeps',
    'beaver':                  'Castor canadensis',
    'belted_kingfisher':       'Megaceryle alcyon',
    'binturong':               'Arctictis binturong',
    'bison':                   'Bison bison',
    'black_bear':              'Ursus americanus',
    'black_vulture':           'Coragyps atratus',
    'blue_heron':              'Ardea herodias',
    'blue_jay':                'Cyanocitta cristata',
    'boa_constrictor':         'Boa constrictor',
    'bobcat':                  'Lynx rufus',
    'bullfrog':                'Lithobates catesbeianus',
    'butterfly':               'Lepidoptera',
    'canada_goose':            'Branta canadensis',
    'capybara':                'Hydrochoerus hydrochaeris',
    'cardinal':                'Cardinalis cardinalis',
    'cassowary':               'Casuarius',
    'cat':                     'Felis catus',
    'chameleon':               'Chamaeleonidae',
    'cheetah':                 'Acinonyx jubatus',
    'chicken':                 'Gallus gallus domesticus',
    'chimpanzee':              'Pan troglodytes',
    'chipmunk':                'Tamias striatus',
    'clownfish':               'Amphiprioninae',
    'cobra':                   'Naja',
    'cockatoo':                'Cacatua',
    'common_raven':            'Corvus corax',
    'cougar':                  'Puma concolor',
    'cow':                     'Bos taurus',
    'coyote':                  'Canis latrans',
    'crab':                    'Cancer pagurus',
    'crane':                   'Grus canadensis',
    'crocodile':               'Crocodylidae',
    'crow':                    'Corvus brachyrhynchos',
    'deer_mouse':              'Peromyscus maniculatus',
    'dingo':                   'Canis lupus dingo',
    'dog':                     'Canis lupus familiaris',
    'dolphin':                 'Delphinidae',
    'dragonfly':               'Anisoptera',
    'echidna':                 'Tachyglossidae',
    'egret':                   'Ardea alba',
    'elephant':                'Elephantidae',
    'elephant_seal':           'Mirounga',
    'emu':                     'Dromaius novaehollandiae',
    'firefly':                 'Lampyridae',
    'flamingo':                'Phoenicopteridae',
    'fox_squirrel':            'Sciurus niger',
    'frog':                    'Rana temporaria',
    'gecko':                   'Gekkonidae',
    'giant_centipede':         'Scolopendra',
    'giant_panda':             'Ailuropoda melanoleuca',
    'giant_squid':             'Architeuthis dux',
    'gila_monster':            'Heloderma suspectum',
    'giraffe':                 'Giraffa',
    'goat':                    'Capra hircus',
    'gorilla':                 'Gorilla',
    'gray_squirrel':           'Sciurus carolinensis',
    'gray_wolf':               'Canis lupus',
    'great_horned_owl':        'Bubo virginianus',
    'great_white_shark':       'Carcharodon carcharias',
    'green_tree_frog':         'Hyla cinerea',
    'grizzly_bear':            'Ursus arctos horribilis',
    'groundhog':               'Marmota monax',
    'hammerhead_shark':        'Sphyrna',
    'hawk':                    'Accipiter',
    'hercules_beetle':         'Dynastes hercules',
    'heron':                   'Ardeidae',
    'hippo':                   'Hippopotamus amphibius',
    'honey_bee':               'Apis mellifera',
    'horse':                   'Equus caballus',
    'horseshoe_crab':          'Limulus polyphemus',
    'house_sparrow':           'Passer domesticus',
    'humpback_whale':          'Megaptera novaeangliae',
    'hummingbird':             'Trochilidae',
    'hyena':                   'Hyaenidae',
    'iguana':                  'Iguana iguana',
    'jaguar':                  'Panthera onca',
    'jellyfish':               'Medusozoa',
    'kangaroo':                'Macropus',
    'killdeer':                'Charadrius vociferus',
    'koala':                   'Phascolarctos cinereus',
    'komodo_dragon':           'Varanus komodoensis',
    'ladybug':                 'Coccinellidae',
    'lemur':                   'Lemur catta',
    'leopard':                 'Panthera pardus',
    'lion':                    'Panthera leo',
    'lobster':                 'Homarus americanus',
    'luna_moth':               'Actias luna',
    'lynx':                    'Lynx canadensis',
    'macaw':                   'Ara macao',
    'mallard':                 'Anas platyrhynchos',
    'manatee':                 'Trichechus',
    'manta_ray':               'Mobula birostris',
    'mantis_shrimp':           'Stomatopoda',
    'meerkat':                 'Suricata suricatta',
    'mockingbird':             'Mimus polyglottos',
    'monarch_butterfly':       'Danaus plexippus',
    'monitor_lizard':          'Varanus',
    'moose':                   'Alces alces',
    'mountain_goat':           'Oreamnos americanus',
    'mourning_dove':           'Zenaida macroura',
    'mule_deer':               'Odocoileus hemionus',
    'numbat':                  'Myrmecobius fasciatus',
    'octopus':                 'Octopus vulgaris',
    'opossum':                 'Didelphis virginiana',
    'orangutan':               'Pongo',
    'orca':                    'Orcinus orca',
    'osprey':                  'Pandion haliaetus',
    'ostrich':                 'Struthio camelus',
    'otter':                   'Lontra canadensis',
    'pangolin':                'Manis',
    'parrot':                  'Psittacidae',
    'peacock':                 'Pavo cristatus',
    'pelican':                 'Pelecanus',
    'penguin':                 'Spheniscidae',
    'peregrine_falcon':        'Falco peregrinus',
    'pig':                     'Sus scrofa domesticus',
    'pigeon':                  'Columba livia',
    'pileated_woodpecker':     'Dryocopus pileatus',
    'platypus':                'Ornithorhynchus anatinus',
    'poison_dart_frog':        'Dendrobatidae',
    'polar_bear':              'Ursus maritimus',
    'porcupine':               'Erethizon dorsatum',
    'praying_mantis':          'Mantodea',
    'puffin':                  'Fratercula arctica',
    'purple_martin':           'Progne subis',
    'rabbit':                  'Oryctolagus cuniculus',
    'raccoon':                 'Procyon lotor',
    'rattlesnake':             'Crotalus',
    'red_eared_slider':        'Trachemys scripta elegans',
    'red_fox':                 'Vulpes vulpes',
    'red_panda':               'Ailurus fulgens',
    'red_tailed_hawk':         'Buteo jamaicensis',
    'red_winged_blackbird':    'Agelaius phoeniceus',
    'rhino':                   'Rhinocerotidae',
    'roseate_spoonbill':       'Platalea ajaja',
    'ruby_throated_hummingbird': 'Archilochus colubris',
    'salamander':              'Caudata',
    'sandhill_crane':          'Antigone canadensis',
    'scarlet_macaw':           'Ara macao',
    'scorpion':                'Scorpiones',
    'sea_lion':                'Zalophus californianus',
    'sea_otter':               'Enhydra lutris',
    'sea_turtle':              'Chelonia mydas',
    'seagull':                 'Larus argentatus',
    'seahorse':                'Hippocampus',
    'seal':                    'Phoca vitulina',
    'secretary_bird':          'Sagittarius serpentarius',
    'shark':                   'Selachimorpha',
    'sheep':                   'Ovis aries',
    'shoebill':                'Balaeniceps rex',
    'skunk':                   'Mephitis mephitis',
    'sloth':                   'Bradypus tridactylus',
    'sloth_bear':              'Melursus ursinus',
    'snake':                   'Serpentes',
    'snapping_turtle':         'Chelydra serpentina',
    'snow_leopard':            'Panthera uncia',
    'snowy_owl':               'Bubo scandiacus',
    'sparrow':                 'Emberizidae',
    'squirrel':                'Sciuridae',
    'starfish':                'Asteroidea',
    'starling':                'Sturnus vulgaris',
    'stingray':                'Dasyatidae',
    'stork':                   'Ciconia ciconia',
    'swan':                    'Cygnus olor',
    'tapir':                   'Tapirus terrestris',
    'tarantula':               'Theraphosidae',
    'tiger':                   'Panthera tigris',
    'tiger_salamander':        'Ambystoma tigrinum',
    'tortoise':                'Testudinidae',
    'toucan':                  'Ramphastidae',
    'turkey':                  'Meleagris gallopavo',
    'turkey_vulture':          'Cathartes aura',
    'turtle':                  'Cheloniidae',
    'vulture':                 'Gyps fulvus',
    'walking_stick':           'Phasmatodea',
    'walrus':                  'Odobenus rosmarus',
    'warthog':                 'Phacochoerus africanus',
    'water_moccasin':          'Agkistrodon piscivorus',
    'whale_shark':             'Rhincodon typus',
    'white_tailed_deer':       'Odocoileus virginianus',
    'whooping_crane':          'Grus americana',
    'wild_boar':               'Sus scrofa',
    'wolf':                    'Canis lupus',
    'wolverine':               'Gulo gulo',
    'wombat':                  'Vombatidae',
    'woodpecker':              'Picidae',
    'yellow_warbler':          'Setophaga petechia',
    'zebra':                   'Equus quagga',
}

LABELS = sorted(INAT_NAMES.keys())
print(f'{len(LABELS)} species:')
for i, l in enumerate(LABELS):
    print(f'  {i:3d}: {l}')

In [ ]:
# ── 3. Download images from iNaturalist ──────────────────────────────────────
import os, requests, time
from pathlib import Path

IMAGES_PER_CLASS = 200
TRAIN_DIR = 'dataset/train'

def download_from_inaturalist(taxon_name, save_dir, max_num=200):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    existing = len(os.listdir(save_dir))
    if existing >= max_num * 0.8:
        print(f'  skip {taxon_name} (already has {existing})')
        return existing

    downloaded = 0
    page = 1
    per_page = 50

    while downloaded < max_num:
        try:
            resp = requests.get(
                'https://api.inaturalist.org/v1/observations',
                params={
                    'taxon_name': taxon_name,
                    'per_page': per_page,
                    'page': page,
                    'photos': 'true',
                    'quality_grade': 'research',
                    'order_by': 'votes',
                },
                headers={'User-Agent': 'WildDex/1.0'},
                timeout=10
            )
            results = resp.json().get('results', [])
            if not results:
                break

            for obs in results:
                if downloaded >= max_num:
                    break
                try:
                    photo_url = obs['photos'][0]['url'].replace('square', 'medium')
                    img = requests.get(photo_url, timeout=8).content
                    if len(img) < 5000:
                        continue
                    with open(f'{save_dir}/{downloaded:04d}.jpg', 'wb') as f:
                        f.write(img)
                    downloaded += 1
                except:
                    pass

            page += 1
            time.sleep(0.5)

        except Exception as e:
            print(f'    error on page {page}: {e}')
            break

    print(f'  {taxon_name}: {downloaded} images')
    return downloaded

for label, taxon in INAT_NAMES.items():
    download_from_inaturalist(taxon, f'{TRAIN_DIR}/{label}', IMAGES_PER_CLASS)

print('\nDownload complete.')

In [ ]:
# ── 4. Verify downloads ──────────────────────────────────────────────────────
print(f'{"Species":<25} {"Count":>6}')
print('-' * 33)
total = 0
low = []
for label in LABELS:
    d = f'{TRAIN_DIR}/{label}'
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    total += n
    flag = ' ⚠️ low' if n < 100 else ''
    print(f'{label:<25} {n:>6}{flag}')
    if n < 100:
        low.append(label)
print(f'\nTotal: {total} images')
if low:
    print(f'Low image classes: {low}')

In [ ]:
# ── 5. Build train/val datasets ──────────────────────────────────────────────
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(LABELS)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
print('Class order matches LABELS:', class_names == LABELS)
print('Classes:', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# ── 6. Build EfficientNetB0 model ────────────────────────────────────────────
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# ── 7. Phase 1: Train head only (base frozen) ────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
]

print('=== Phase 1: Training head (base frozen) ===')
h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

In [ ]:
# ── 8. Phase 2: Fine-tune top layers ─────────────────────────────────────────
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks2 = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
]

print('=== Phase 2: Fine-tuning top layers ===')
h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks2)

model.save('wilddex_model.keras')
print('Model saved.')

In [ ]:
# ── 9. Plot training results ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

acc  = h1.history['accuracy']     + h2.history['accuracy']
vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss = h1.history['loss']         + h2.history['loss']
vloss= h1.history['val_loss']     + h2.history['val_loss']
split = len(h1.history['accuracy'])
epochs = range(len(acc))

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(epochs, acc,  label='Train')
plt.plot(epochs, vacc, label='Val')
plt.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Accuracy'); plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, loss,  label='Train')
plt.plot(epochs, vloss, label='Val')
plt.axvline(split, color='gray', linestyle='--', label='Fine-tune start')
plt.title('Loss'); plt.legend()
plt.tight_layout()
plt.savefig('training_results.png')
plt.show()

val_loss, val_acc = model.evaluate(val_ds)
print(f'\nFinal val accuracy: {val_acc*100:.1f}%')

In [ ]:
# ── 10. Convert to TFLite (float16) ──────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('wilddex_model.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'TFLite model saved — {size_mb:.1f} MB')

In [ ]:
# ── 11. Save labels.txt ───────────────────────────────────────────────────────
with open('labels.txt', 'w') as f:
    f.write('\n'.join(LABELS))

print('labels.txt:')
print('\n'.join(f'{i}: {l}' for i, l in enumerate(LABELS)))

In [ ]:
# ── 12. Download files ────────────────────────────────────────────────────────
from google.colab import files
files.download('wilddex_model.tflite')
files.download('labels.txt')
files.download('training_results.png')
print('Done! Add wilddex_model.tflite and labels.txt to your app.')